# Phase 0: GSM8K sanity check, base vs MIT-edited Qwen3.5-0.8B

Reproduces the math check in PLAN.md §3.4: **100 GSM8K test problems** (the first 100 of the test split), thinking on,
the paper's sampling (temp 0.6, top_p 0.95, top_k 20, presence_penalty 1.5, at most 32,768 new tokens, seed 0),
graded on the last `\boxed{}` after `</think>` with math-verify.

**Runtime:** use a GPU runtime (Runtime → Change runtime type). **A100** or **L4** is best: they support bf16, like the
paper's evaluation. A T4 works too, but it runs in fp16 and is much slower.

Rough times for SAMPLES=1 (about 5M generated tokens in total; the base model loops to the token limit almost every
time): A100 about 20–40 min, L4 about 1–1.5 h, T4 several hours.

Results are written to Google Drive (if enabled), so a disconnected session can resume: a model whose `summary_*.json`
already exists is skipped.

In [ ]:
# ---- settings ----
N_PROBLEMS = 100        # first N problems of the GSM8K test split
SAMPLES = 1             # samples per problem (the paper used 2 for GSM8K; 2 doubles the time)
MAX_TOKENS = 32768      # the paper's limit
SAVE_TO_DRIVE = True    # write results to /content/drive/MyDrive/sql-wsp/results/p0_gsm8k100

BASE_REPO, BASE_REV = "Qwen/Qwen3.5-0.8B", "2fc06364715b967f1860aea9cf38778875588b17"
EDIT_REPO, EDIT_REV = "RikiyaT/Qwen3.5-0.8B-weight-space-programming", "d1671a178705bcab7644806d23c535517f35723b"

import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv"],
                     capture_output=True, text=True).stdout)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/sql-wsp/results/p0_gsm8k100"
else:
    OUT = "/content/results/p0_gsm8k100"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 1. Install (vLLM 0.33.0 / transformers 5.20.2, the versions tested locally). Takes a few minutes.

In [ ]:
!pip install -q "vllm==0.33.0" "transformers==5.20.2" "math-verify[antlr4_13_2]" datasets
import vllm, transformers, torch
# is_bf16_supported() is True on a T4 too (emulated); vLLM needs native bf16, i.e. compute capability >= 8.0
print(vllm.__version__, transformers.__version__, torch.__version__, torch.cuda.get_device_name(0),
      "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16 only (no native bf16)")

## 2. Download both models at pinned revisions and verify the edit

The expected sha256 values come from `exp8_math/runs/p50c/edit/manifest.json` in `RikiyaT/lm_understand-logs`. The
check also confirms that the edited checkpoint differs from the base in exactly these two tensors. Both repos contain
only safetensors files and configs, with no custom code, so `trust_remote_code` is not needed.

In [ ]:
import hashlib, json, struct
from huggingface_hub import snapshot_download

BASE = snapshot_download(BASE_REPO, revision=BASE_REV, local_dir="/content/models/Qwen3.5-0.8B")
EDIT = snapshot_download(EDIT_REPO, revision=EDIT_REV, local_dir="/content/models/Qwen3.5-0.8B-wsp",
                         allow_patterns=["*.json", "*.jinja", "*.txt", "*.safetensors", "LICENSE"])
ST = "model.safetensors-00001-of-00001.safetensors"
EXPECTED = {  # name: (base sha256, edited sha256)
    "model.language_model.layers.15.mlp.down_proj.weight":
        ("ef60d2c687bab19464dca9c8bfd7777e46ad226df0c9e25643b0bab04b5a1cca",
         "82e2d733ea64feaeb4af93a1e065641d3df98b494a8b7576ef2f52e90fe2ee53"),
    "model.language_model.layers.19.mlp.down_proj.weight":
        ("638eedeb701b94aa861f4438e615c07ec6c60ee703db1bfee2de577bb6498c3e",
         "8ef3c61bc36af113ad58e89dbb0a4a14dcba5a45cb750c15ab515c82ddd12eb2"),
}

def tensor_hashes(path):
    # sha256 of each tensor's raw bytes (bf16 bits), read straight from the safetensors file
    out = {}
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        head = json.loads(f.read(n))
        for name, info in head.items():
            if name == "__metadata__":
                continue
            a, b = info["data_offsets"]
            f.seek(8 + n + a)
            out[name] = hashlib.sha256(f.read(b - a)).hexdigest()
    return out

hb, he = tensor_hashes(os.path.join(BASE, ST)), tensor_hashes(os.path.join(EDIT, ST))
diff = sorted(k for k in hb if hb[k] != he.get(k))
print(len(hb), "tensors; differing:", diff)
assert set(hb) == set(he), "tensor names differ"
assert diff == sorted(EXPECTED), "unexpected set of edited tensors"
for name, (b, e) in EXPECTED.items():
    assert hb[name] == b and he[name] == e, name
    print("hash OK:", name)
json.dump({"base": {k: hb[k] for k in EXPECTED}, "edited": {k: he[k] for k in EXPECTED}, "differing": diff},
          open(os.path.join(OUT, "hash_check.json"), "w"), indent=1)

## 3. GSM8K test problems

In [ ]:
from datasets import load_dataset
ds = load_dataset("openai/gsm8k", "main", split="test")
PROBLEMS = "/content/gsm8k_test.jsonl"
with open(PROBLEMS, "w") as f:
    for i, r in enumerate(ds.select(range(N_PROBLEMS))):
        f.write(json.dumps({"id": f"gsm8k_test_{i}", "problem": r["question"],
                            "answer": r["answer"].split("####")[-1].strip().replace(",", "")}) + "\n")
print(len(ds), "test problems; using", N_PROBLEMS)

## 4. Evaluation script (same protocol as `src/gsm8k_check.py` in the project)

In [ ]:
%%writefile /content/gsm8k_check.py
"""Phase 0 math sanity check: GSM8K test problems, base vs edited Qwen3.5-0.8B, thinking on, the paper's sampling.

Runs inside WSL with vLLM. Protocol follows ref/.../exp8_math/code/evaluate.py (reimplemented, not imported):
user turn = problem + "\\n" + "Please reason step by step, and put your final answer within \\boxed{}.",
chat template with enable_thinking=True, temp 0.6 / top_p 0.95 / top_k 20 / presence_penalty 1.5, seed 0,
graded on the last \\boxed{} after the last </think> with math-verify.

    python src/gsm8k_check.py --model <dir> --tag base --n-problems 10 --max-tokens 32768 --out results/<run_id>
"""
import argparse
import json
import os
import time

SUFFIX = "Please reason step by step, and put your final answer within \\boxed{}."


def last_boxed(text):
    i = text.rfind("\\boxed{")
    if i < 0:
        return None
    j, depth = i + len("\\boxed{"), 1
    for k in range(j, len(text)):
        depth += {"{": 1, "}": -1}.get(text[k], 0)
        if depth == 0:
            return text[j:k]
    return None


def tail_dup8(text, chars=2000):
    """share of duplicated word 8-grams in the last `chars` characters (like mean_cut_tail_duplicate8)."""
    w = text[-chars:].split()
    grams = [tuple(w[i:i + 8]) for i in range(len(w) - 7)]
    return 1 - len(set(grams)) / len(grams) if grams else 0.0


def grade(pred, gold):
    from math_verify import parse, verify
    if pred is None:
        return False
    try:
        return bool(verify(parse("$" + gold + "$"), parse("$" + pred + "$")))
    except Exception:
        return pred.strip().replace(",", "") == gold


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--tag", required=True)
    ap.add_argument("--problems", required=True, help="jsonl with id, problem, answer")
    ap.add_argument("--n-problems", type=int, default=100)
    ap.add_argument("--samples", type=int, default=1)
    ap.add_argument("--max-tokens", type=int, default=32768)
    ap.add_argument("--dtype", default="float16")
    ap.add_argument("--mem", type=float, default=0.80)
    ap.add_argument("--out", required=True)
    a = ap.parse_args()

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    rows = [json.loads(l) for l in open(a.problems)][:a.n_problems]
    tok = AutoTokenizer.from_pretrained(a.model)
    prompts = [tok.apply_chat_template([{"role": "user", "content": r["problem"] + "\n" + SUFFIX}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=True) for r in rows]
    sampling = {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "presence_penalty": 1.5,
                "max_tokens": a.max_tokens, "seed": 0}
    llm = LLM(model=a.model, dtype=a.dtype, gpu_memory_utilization=a.mem, max_model_len=a.max_tokens + 2048,
              seed=0, limit_mm_per_prompt={"image": 0, "video": 0})
    t0 = time.time()
    outs = llm.generate(prompts, SamplingParams(n=a.samples, **sampling))
    wall = time.time() - t0

    os.makedirs(a.out, exist_ok=True)
    per, ok, cut, toks, dups = [], 0, 0, [], []
    with open(os.path.join(a.out, f"per_item_{a.tag}.jsonl"), "w") as f:
        for r, o in zip(rows, outs):
            for s_i, s in enumerate(o.outputs):
                text = s.text
                answer_part = text.rsplit("</think>", 1)[-1] if "</think>" in text else ""
                pred = last_boxed(answer_part)
                good = grade(pred, r["answer"])
                is_cut = s.finish_reason == "length"
                ok += good
                cut += is_cut
                toks.append(len(s.token_ids))
                if is_cut:
                    dups.append(tail_dup8(text))
                f.write(json.dumps({"id": r["id"], "sample": s_i, "gold": r["answer"], "pred": pred, "correct": good,
                                    "cut": is_cut, "think_closed": "</think>" in text, "tokens": len(s.token_ids),
                                    "tail_dup8": tail_dup8(text), "text": text}) + "\n")
    n = len(toks)
    summary = {"tag": a.tag, "model": a.model, "n_problems": len(rows), "samples": a.samples, "acc": ok / n,
               "cut_share": cut / n, "mean_tokens": sum(toks) / n, "tokens_sum": sum(toks),
               "mean_cut_tail_duplicate8": sum(dups) / len(dups) if dups else None, "wall_s": wall,
               "tok_per_s": sum(toks) / wall, "dtype": a.dtype, "sampling": sampling}
    json.dump(summary, open(os.path.join(a.out, f"summary_{a.tag}.json"), "w"), indent=1)
    print(json.dumps(summary, indent=1))


if __name__ == "__main__":
    main()

## 5. Run base, then edited

Each model runs in its own process, so GPU memory is freed between runs. A model whose summary already exists in
`OUT` is skipped (resume after a disconnect).

In [ ]:
import torch
DTYPE = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"   # T4 (7.5) -> float16
print("dtype:", DTYPE)
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"

def run(tag, model):
    if os.path.exists(os.path.join(OUT, f"summary_{tag}.json")):
        print(tag, "already done, skipping")
        return
    cmd = ["python", "/content/gsm8k_check.py", "--model", model, "--tag", tag, "--problems", PROBLEMS,
           "--n-problems", str(N_PROBLEMS), "--samples", str(SAMPLES), "--max-tokens", str(MAX_TOKENS),
           "--dtype", DTYPE, "--mem", "0.90", "--out", OUT]
    with open(os.path.join(OUT, f"log_{tag}.txt"), "w") as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:          # stream the progress bar and the final summary into the cell
            print(line, end="")
            log.write(line)
        assert p.wait() == 0, f"{tag} run failed, see log_{tag}.txt"

run("base", BASE)

In [ ]:
run("edited", EDIT)

## 6. Results: GATE 0 numbers

In [ ]:
import pandas as pd
S = {t: json.load(open(os.path.join(OUT, f"summary_{t}.json"))) for t in ("base", "edited")}
P = {t: {(r["id"], r["sample"]): r for r in map(json.loads, open(os.path.join(OUT, f"per_item_{t}.jsonl")))}
     for t in ("base", "edited")}
rows = []
for t, s in S.items():
    closed = sum(r["think_closed"] for r in P[t].values()) / len(P[t])
    rows.append({"model": t, "acc": s["acc"], "cut_share": s["cut_share"], "think_closed": closed,
                 "mean_tokens": round(s["mean_tokens"]), "cut_tail_dup8": s["mean_cut_tail_duplicate8"],
                 "gpu_hours": round(s["wall_s"] / 3600, 2), "tok_per_s": round(s["tok_per_s"]), "dtype": s["dtype"]})
table = pd.DataFrame(rows).set_index("model")
display(table)

keys = sorted(set(P["base"]) & set(P["edited"]))
rescues = [k for k in keys if P["edited"][k]["correct"] and not P["base"][k]["correct"]]
losses = [k for k in keys if P["base"][k]["correct"] and not P["edited"][k]["correct"]]
rescued_cut = sum(P["base"][k]["cut"] for k in rescues)
gate = {"n": len(keys), "acc_base": S["base"]["acc"], "acc_edited": S["edited"]["acc"],
        "gain": S["edited"]["acc"] - S["base"]["acc"], "rescues": len(rescues), "losses": len(losses),
        "rescues_where_base_was_cut": rescued_cut,
        "paper_gsm8k": {"unedited": 0.138, "edited_p50c": 0.671},
        "gpu_hours_total": sum(s["wall_s"] for s in S.values()) / 3600, "gpu": torch.cuda.get_device_name(0)}
print(json.dumps(gate, indent=1))
json.dump({"table": rows, "gate0": gate}, open(os.path.join(OUT, "gate0_summary.json"), "w"), indent=1)

## 7. Download the results (or copy the Drive folder `sql-wsp/results/p0_gsm8k100` into the project's `results/`)

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/p0_gsm8k100", "zip", OUT)
from google.colab import files
files.download(zip_path)